In [ ]:
# Block 1

#Notebook description:

# this notebook is used to evaluate the market of assets and find potential assets to invest in
# with the best risk/reward characteristics. This notebook is intented to analyze a broad group of market assets
# and does NOT focus on any particular assets. Computing data for a large number of assets is computationally expensive and thus
# the analysis is relegated to other notebooks


In [ ]:
# Block 2

#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys
import sys
from dotenv import load_dotenv
import os

cwd = os.getcwd()
project_path = cwd
while not os.path.exists(os.path.join(project_path, "pyproject.toml")):
    parent = os.path.dirname(project_path)
    if parent == project_path:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    project_path = parent
if project_path not in sys.path:
    sys.path.append(project_path)
load_dotenv(os.path.join(project_path, ".env"))

from Quantapp.visualization import Plotter
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper
from Quantapp.data import MacroDataClient
from Quantapp.data import MarketDataClient
from Quantapp.data import GICSDataClient
from Quantapp.visualization import Plotter

import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import holidays
import plotly.express as px
import concurrent.futures

#shut down warnings
import warnings
warnings.filterwarnings("ignore")

time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 365
interval = '1d'
period     = '10y'

risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
benchmark = 'SPY'

#qc = Rolling()
rolling = Rolling()
qp = Plotter()
ed = MacroDataClient()
#helper = Helper()
md = MarketDataClient()
gics_data = GICSDataClient(save_path=project_path)



In [ ]:
# Block 3

#define functions



#function to calculate rolling Sharpe ratio
def calculate_rolling_sharpe(df, window=21, risk_free_rate=0):
    daily_returns = df.pct_change()
    rolling_mean = daily_returns.rolling(window=window, min_periods=1).mean()
    rolling_std = daily_returns.rolling(window=window, min_periods=1).std()
    rolling_sharpe = (rolling_mean - risk_free_rate) / rolling_std
    return rolling_sharpe

#function to calculate rolling Sharpe ratio z-score
def calculate_rolling_sharpe_zscore(df, window=21, z_window=None, risk_free_rate=0):
    if z_window is None:
        z_window = window
    rolling_sharpe = calculate_rolling_sharpe(df, window, risk_free_rate)
    rolling_mean = rolling_sharpe.rolling(window=z_window, min_periods=1).mean()
    rolling_std = rolling_sharpe.rolling(window=z_window, min_periods=1).std()
    z_score = (rolling_sharpe - rolling_mean) / rolling_std
    return z_score

#function to count number of stocks above/below certain Sharpe ratio z-score thresholds
def count_stocks_above_sharpe_zscore(df, above_or_below='both', zscore_thresholds=[0,1,2,3], window=21, z_window=None, risk_free_rate=0):
    rolling_sharpe_zscore = calculate_rolling_sharpe_zscore(df, window, z_window, risk_free_rate)
    results = {}
    
    for threshold in zscore_thresholds:
        if above_or_below in ['above', 'both']:
            count_above = (rolling_sharpe_zscore > threshold).sum(axis=1)
            results[f'Count Above Z-Score {threshold} (window: {window})'] = count_above
        if above_or_below in ['below', 'both']:
            count_below = (rolling_sharpe_zscore < -threshold).sum(axis=1)
            results[f'Count Below Z-Score -{threshold} (window: {window})'] = count_below
    
    return pd.DataFrame(results)

#function to count number of stocks above/below benchmark Sharpe
def count_stocks_above_benchmark_sharpe(df, benchmark_series, window=21, risk_free_rate=0):
    # Ensure both indexes are tz-naive
    if df.index.tz:
        df.index = df.index.tz_localize(None)
    if benchmark_series.index.tz:
        benchmark_series.index = benchmark_series.index.tz_localize(None)
    
    # Calculate rolling Sharpe
    rolling_sharpe = calculate_rolling_sharpe(df, window, risk_free_rate)
    benchmark_sharpe = calculate_rolling_sharpe(benchmark_series, window, risk_free_rate)
    
    # Reindex benchmark to match stocks
    benchmark_sharpe = benchmark_sharpe.reindex(rolling_sharpe.index)
    
    # Count stocks above/below benchmark
    count_above = (rolling_sharpe.gt(benchmark_sharpe, axis=0)).sum(axis=1)
    count_below = (rolling_sharpe.lt(benchmark_sharpe, axis=0)).sum(axis=1)
    
    return pd.DataFrame({
        'Count Above Benchmark Sharpe': count_above,
        'Count Below Benchmark Sharpe': count_below,
        'Net Count Above Benchmark Sharpe': count_above - count_below
    }, index=rolling_sharpe.index)


In [ ]:
# Block 4

#parameters
#valid values for capitalization: 'Large Cap', 'Mid Cap', 'Small Cap', 
selected_capitalization = 'Large Cap'

In [ ]:
# Block 5

#load all companies
all_companies = gics_data.retrieve_companies()

selected_companies_large_cap = all_companies[all_companies['Capitalization'] == 'Large Cap']
benchmark_large_cap = yf.Ticker('SPY').history(period=period, interval=interval)['Close']

selected_companies_mid_cap = all_companies[all_companies['Capitalization'] == 'Mid Cap']
benchmark_mid_cap = yf.Ticker('MDY').history(period=period, interval=interval)['Close']

selected_companies_small_cap = all_companies[all_companies['Capitalization'] == 'Small Cap']
benchmark_small_cap = yf.Ticker('IJR').history(period=period, interval=interval)['Close']
    
company_list_large_cap = all_companies[all_companies['Capitalization'] == 'Large Cap']['Symbol'].tolist()
company_list_mid_cap = all_companies[all_companies['Capitalization'] == 'Mid Cap']['Symbol'].tolist()
company_list_small_cap = all_companies[all_companies['Capitalization'] == 'Small Cap']['Symbol'].tolist()
    

companies_data_large_cap = md.generate_series(company_list_large_cap, columns=['Close'], period=period, interval=interval)
companies_data_mid_cap = md.generate_series(company_list_mid_cap, columns=['Close'], period=period, interval=interval)
companies_data_small_cap = md.generate_series(company_list_small_cap, columns=['Close'], period=period, interval=interval)


In [ ]:
# Block 6

#Filter by Sectors
healthcare_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Health Care']['Symbol'].tolist()]
technology_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Information Technology']['Symbol'].tolist()]
financials_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Financials']['Symbol'].tolist()]
consumer_discretionary_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Consumer Discretionary']['Symbol'].tolist()]
consumer_staples_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Consumer Staples']['Symbol'].tolist()]
energy_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Energy']['Symbol'].tolist()]
industrials_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Industrials']['Symbol'].tolist()]
materials_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Materials']['Symbol'].tolist()]
real_estate_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Real Estate']['Symbol'].tolist()]
utilities_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Utilities']['Symbol'].tolist()]
communication_services_companies_data_large_cap = companies_data_large_cap[selected_companies_large_cap[selected_companies_large_cap['Sector'] == 'Communication Services']['Symbol'].tolist()] 

healthcare_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Health Care']['Symbol'].tolist()]
technology_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Information Technology']['Symbol'].tolist()]
financials_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Financials']['Symbol'].tolist()]
consumer_discretionary_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Consumer Discretionary']['Symbol'].tolist()]
consumer_staples_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Consumer Staples']['Symbol'].tolist()]
energy_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Energy']['Symbol'].tolist()]
industrials_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Industrials']['Symbol'].tolist()]
materials_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Materials']['Symbol'].tolist()]
real_estate_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Real Estate']['Symbol'].tolist()]
utilities_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Utilities']['Symbol'].tolist()]
communication_services_companies_data_mid_cap = companies_data_mid_cap[selected_companies_mid_cap[selected_companies_mid_cap['Sector'] == 'Communication Services']['Symbol'].tolist()] 

healthcare_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Health Care']['Symbol'].tolist()]
technology_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Information Technology']['Symbol'].tolist()]
financials_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Financials']['Symbol'].tolist()]
consumer_discretionary_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Consumer Discretionary']['Symbol'].tolist()]
consumer_staples_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Consumer Staples']['Symbol'].tolist()]
energy_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Energy']['Symbol'].tolist()]
industrials_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Industrials']['Symbol'].tolist()]
materials_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Materials']['Symbol'].tolist()]
real_estate_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Real Estate']['Symbol'].tolist()]
utilities_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Utilities']['Symbol'].tolist()]
communication_services_companies_data_small_cap = companies_data_small_cap[selected_companies_small_cap[selected_companies_small_cap['Sector'] == 'Communication Services']['Symbol'].tolist()] 

In [ ]:
# Block 7

#calculate stocks above/below sharpe ratio z-score thresholds and benchmark sharpe

#first calculate for all selected companies
count_stocks_above_below_sharpe_zscore_200 = count_stocks_above_sharpe_zscore(companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_benchmark_all = count_stocks_above_benchmark_sharpe(companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_benchmark_sharpe_200    = count_stocks_above_benchmark_sharpe(companies_data_large_cap, benchmark_large_cap, window=200)

#------------------------------------------------------------------------------------------------------------------------------------
count_stocks_above_below_sharpe_zscore_200_healthcare_large_cap = count_stocks_above_sharpe_zscore(healthcare_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_technology_large_cap = count_stocks_above_sharpe_zscore(technology_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_financials_large_cap = count_stocks_above_sharpe_zscore(financials_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200) 
count_stocks_above_below_sharpe_zscore_200_consumer_discretionary_large_cap = count_stocks_above_sharpe_zscore(consumer_discretionary_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_consumer_staples_large_cap = count_stocks_above_sharpe_zscore(consumer_staples_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_energy_large_cap = count_stocks_above_sharpe_zscore(energy_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)     
count_stocks_above_below_sharpe_zscore_200_industrials_large_cap = count_stocks_above_sharpe_zscore(industrials_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_materials_large_cap = count_stocks_above_sharpe_zscore(materials_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_real_estate_large_cap = count_stocks_above_sharpe_zscore(real_estate_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_utilities_large_cap = count_stocks_above_sharpe_zscore(utilities_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_communication_services_large_cap = count_stocks_above_sharpe_zscore(communication_services_companies_data_large_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)

count_stocks_above_below_sharpe_benchmark_healthcare_large_cap = count_stocks_above_benchmark_sharpe(healthcare_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_technology_large_cap = count_stocks_above_benchmark_sharpe(technology_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_financials_large_cap = count_stocks_above_benchmark_sharpe(financials_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_discretionary_large_cap = count_stocks_above_benchmark_sharpe(consumer_discretionary_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_staples_large_cap = count_stocks_above_benchmark_sharpe(consumer_staples_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_energy_large_cap = count_stocks_above_benchmark_sharpe(energy_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_industrials_large_cap = count_stocks_above_benchmark_sharpe(industrials_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_materials_large_cap = count_stocks_above_benchmark_sharpe(materials_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_real_estate_large_cap = count_stocks_above_benchmark_sharpe(real_estate_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_utilities_large_cap = count_stocks_above_benchmark_sharpe(utilities_companies_data_large_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_communication_services_large_cap = count_stocks_above_benchmark_sharpe(communication_services_companies_data_large_cap, benchmark_large_cap, window=200)
#------------------------------------------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------------------------------------------
count_stocks_above_below_sharpe_zscore_200_healthcare_mid_cap = count_stocks_above_sharpe_zscore(healthcare_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_technology_mid_cap = count_stocks_above_sharpe_zscore(technology_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_financials_mid_cap = count_stocks_above_sharpe_zscore(financials_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200) 
count_stocks_above_below_sharpe_zscore_200_consumer_discretionary_mid_cap = count_stocks_above_sharpe_zscore(consumer_discretionary_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_consumer_staples_mid_cap = count_stocks_above_sharpe_zscore(consumer_staples_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_energy_mid_cap = count_stocks_above_sharpe_zscore(energy_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)     
count_stocks_above_below_sharpe_zscore_200_industrials_mid_cap = count_stocks_above_sharpe_zscore(industrials_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_materials_mid_cap = count_stocks_above_sharpe_zscore(materials_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_real_estate_mid_cap = count_stocks_above_sharpe_zscore(real_estate_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_utilities_mid_cap = count_stocks_above_sharpe_zscore(utilities_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_communication_services_mid_cap = count_stocks_above_sharpe_zscore(communication_services_companies_data_mid_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)

count_stocks_above_below_sharpe_benchmark_healthcare_mid_cap = count_stocks_above_benchmark_sharpe(healthcare_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_technology_mid_cap = count_stocks_above_benchmark_sharpe(technology_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_financials_mid_cap = count_stocks_above_benchmark_sharpe(financials_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_discretionary_mid_cap = count_stocks_above_benchmark_sharpe(consumer_discretionary_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_staples_mid_cap = count_stocks_above_benchmark_sharpe(consumer_staples_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_energy_mid_cap = count_stocks_above_benchmark_sharpe(energy_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_industrials_mid_cap = count_stocks_above_benchmark_sharpe(industrials_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_materials_mid_cap = count_stocks_above_benchmark_sharpe(materials_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_real_estate_mid_cap = count_stocks_above_benchmark_sharpe(real_estate_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_utilities_mid_cap = count_stocks_above_benchmark_sharpe(utilities_companies_data_mid_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_communication_services_mid_cap = count_stocks_above_benchmark_sharpe(communication_services_companies_data_mid_cap, benchmark_large_cap, window=200)
#------------------------------------------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------------------------------------------
count_stocks_above_below_sharpe_zscore_200_healthcare_small_cap = count_stocks_above_sharpe_zscore(healthcare_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_technology_small_cap = count_stocks_above_sharpe_zscore(technology_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_financials_small_cap = count_stocks_above_sharpe_zscore(financials_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200) 
count_stocks_above_below_sharpe_zscore_200_consumer_discretionary_small_cap = count_stocks_above_sharpe_zscore(consumer_discretionary_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_consumer_staples_small_cap = count_stocks_above_sharpe_zscore(consumer_staples_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)
count_stocks_above_below_sharpe_zscore_200_energy_small_cap = count_stocks_above_sharpe_zscore(energy_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)     
count_stocks_above_below_sharpe_zscore_200_industrials_small_cap = count_stocks_above_sharpe_zscore(industrials_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_materials_small_cap = count_stocks_above_sharpe_zscore(materials_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_real_estate_small_cap = count_stocks_above_sharpe_zscore(real_estate_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_utilities_small_cap = count_stocks_above_sharpe_zscore(utilities_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)   
count_stocks_above_below_sharpe_zscore_200_communication_services_small_cap = count_stocks_above_sharpe_zscore(communication_services_companies_data_small_cap, above_or_below='both', zscore_thresholds=[0,1,2,3], window=200)

count_stocks_above_below_sharpe_benchmark_healthcare_small_cap = count_stocks_above_benchmark_sharpe(healthcare_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_technology_small_cap = count_stocks_above_benchmark_sharpe(technology_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_financials_small_cap = count_stocks_above_benchmark_sharpe(financials_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_discretionary_small_cap = count_stocks_above_benchmark_sharpe(consumer_discretionary_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_consumer_staples_small_cap = count_stocks_above_benchmark_sharpe(consumer_staples_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_energy_small_cap = count_stocks_above_benchmark_sharpe(energy_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_industrials_small_cap = count_stocks_above_benchmark_sharpe(industrials_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_materials_small_cap = count_stocks_above_benchmark_sharpe(materials_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_real_estate_small_cap = count_stocks_above_benchmark_sharpe(real_estate_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_utilities_small_cap = count_stocks_above_benchmark_sharpe(utilities_companies_data_small_cap, benchmark_large_cap, window=200)
count_stocks_above_below_sharpe_benchmark_communication_services_small_cap = count_stocks_above_benchmark_sharpe(communication_services_companies_data_small_cap, benchmark_large_cap, window=200)
#------------------------------------------------------------------------------------------------------------------------------------


In [ ]:
# Block 8

import pandas as pd
import plotly.graph_objects as go


def normalize_x_index(index):
    idx = pd.Index(index)

    if isinstance(idx, pd.PeriodIndex):
        return idx.to_timestamp()

    if pd.api.types.is_datetime64_any_dtype(idx):
        return pd.to_datetime(idx)

    parsed = pd.to_datetime(idx, errors="coerce")
    if len(parsed) > 0 and parsed.notna().all():
        return parsed

    return idx


def plot_net_count_dropdown_with_stats(sector_count_dfs, default_sector="All"):

    fig = go.Figure()

    sector_names = list(sector_count_dfs.keys())

    if default_sector not in sector_names:
        default_sector = sector_names[0]

    stats = {}
    trace_indices = {}
    annotations_per_sector = {}

    trace_counter = 0

    for sector, df in sector_count_dfs.items():

        series = df["Net Count Above Benchmark Sharpe"]
        x_values = normalize_x_index(series.index)

        mean = series.mean()
        std = series.std()

        levels = [
            (mean, "Mean"),
            (mean + std, "+1 sigma"),
            (mean - std, "-1 sigma"),
            (mean + 2 * std, "+2 sigma"),
            (mean - 2 * std, "-2 sigma"),
            (mean + 3 * std, "+3 sigma"),
            (mean - 3 * std, "-3 sigma"),
        ]

        stats[sector] = {"series": series, "levels": levels}

        sector_visible = sector == default_sector

        trace_indices[sector] = []

        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=series,
                mode="lines",
                name=sector,
                visible=sector_visible,
            )
        )
        trace_indices[sector].append(trace_counter)
        trace_counter += 1

        annotations = []

        for level, label in levels:

            fig.add_trace(
                go.Scatter(
                    x=x_values,
                    y=[level] * len(series),
                    mode="lines",
                    line=dict(dash="dash"),
                    showlegend=False,
                    visible=sector_visible,
                )
            )

            trace_indices[sector].append(trace_counter)
            trace_counter += 1

            annotations.append(
                dict(
                    x=x_values[-1],
                    y=level,
                    xref="x",
                    yref="y",
                    text=label,
                    showarrow=False,
                    xanchor="left",
                    font=dict(size=12),
                )
            )

        annotations_per_sector[sector] = annotations

    buttons = []

    total_traces = trace_counter

    for sector in sector_names:

        visible = [False] * total_traces

        for idx in trace_indices[sector]:
            visible[idx] = True

        buttons.append(
            dict(
                label=sector,
                method="update",
                args=[
                    {"visible": visible},
                    {
                        "title": f"Net Count Above Benchmark Sharpe - {sector}",
                        "annotations": annotations_per_sector[sector],
                    },
                ],
            )
        )

    fig.update_layout(
        title=f"Net Count Above Benchmark Sharpe - {default_sector}",
        xaxis_title="Date",
        yaxis_title="Net Count",
        xaxis=dict(type="date"),
        annotations=annotations_per_sector[default_sector],
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=0.01,
                y=1.15,
                xanchor="left",
                yanchor="top",
            )
        ],
    )

    return fig


def plot_sharpe_zscore_by_sector(
    sector_data,
    thresholds_list,
    selected_cap,
    sigmas=(1, 2, 3),
    show_mean=True,
    ddof=0,
    height=600,
):
    """
    Jupyter interactive plot:

    - Dropdown 1: Sector
    - Dropdown 2: Threshold
    - Displays one line at a time
    - Adds mean line (optional)
    - Adds only positive sigma lines
    - Red shading between 1.5 sigma and 2 sigma
    """

    sectors = list(sector_data.keys())
    num_thresholds = len(thresholds_list)

    fig = go.FigureWidget()

    for sector_name, df in sector_data.items():
        x_values = normalize_x_index(df.index)
        for thresh in thresholds_list:
            fig.add_trace(
                go.Scatter(
                    x=x_values,
                    y=df[thresh],
                    mode="lines",
                    visible=False,
                    name=f"{sector_name} - {thresh}",
                    hovertemplate=(
                        f"{sector_name}"
                        f"<br>Threshold: {thresh}"
                        f"<br>Date: %{{x}}"
                        f"<br>Count: %{{y}}"
                        f"<extra></extra>"
                    ),
                )
            )

    def trace_index(sector_idx, thresh_idx):
        return sector_idx * num_thresholds + thresh_idx

    sector_dd = Dropdown(options=sectors, value=sectors[0], description="Sector:")
    thresh_dd = Dropdown(options=thresholds_list, value=thresholds_list[0], description="Threshold:")

    def build_sigma_shapes_and_labels(x0, x1, mu, sd):

        shapes = []
        annotations = []

        if show_mean:
            shapes.append(
                dict(
                    type="line",
                    xref="x",
                    yref="y",
                    x0=x0,
                    x1=x1,
                    y0=mu,
                    y1=mu,
                    line=dict(width=2),
                )
            )

            annotations.append(
                dict(
                    xref="paper",
                    yref="y",
                    x=0.99,
                    y=mu,
                    text=f"mu = {mu:.2f}",
                    showarrow=False,
                    xanchor="right",
                    bgcolor="rgba(255,255,255,0.7)",
                )
            )

        y_1_5_sigma = mu + 1.5 * sd
        y_2_sigma = mu + 2 * sd

        shapes.append(
            dict(
                type="rect",
                xref="x",
                yref="y",
                x0=x0,
                x1=x1,
                y0=y_1_5_sigma,
                y1=y_2_sigma,
                fillcolor="rgba(255,0,0,0.15)",
                line=dict(width=0),
                layer="below",
            )
        )

        for sigma in sigmas:

            y_level = mu + sigma * sd

            shapes.append(
                dict(
                    type="line",
                    xref="x",
                    yref="y",
                    x0=x0,
                    x1=x1,
                    y0=y_level,
                    y1=y_level,
                    line=dict(width=1, dash="dash"),
                )
            )

            annotations.append(
                dict(
                    xref="paper",
                    yref="y",
                    x=0.99,
                    y=y_level,
                    text=f"+{sigma} sigma ({y_level:.2f})",
                    showarrow=False,
                    xanchor="right",
                    bgcolor="rgba(255,255,255,0.7)",
                )
            )

        return shapes, annotations

    def apply_selection(*_):

        s_idx = sectors.index(sector_dd.value)
        t_idx = thresholds_list.index(thresh_dd.value)
        ti = trace_index(s_idx, t_idx)

        vis = [False] * len(fig.data)
        vis[ti] = True

        df = sector_data[sector_dd.value]
        x_values = normalize_x_index(df.index)
        y = np.asarray(df[thresh_dd.value].values, dtype=float)
        y_clean = y[~np.isnan(y)]

        if len(x_values) > 0:
            x0, x1 = x_values.min(), x_values.max()
        else:
            x0, x1 = 0, 1

        if y_clean.size == 0:
            mu, sd = 0.0, 0.0
        else:
            mu = float(np.mean(y_clean))
            sd = float(np.std(y_clean, ddof=ddof))

        shapes, annotations = build_sigma_shapes_and_labels(x0, x1, mu, sd)

        with fig.batch_update():

            for index, trace in enumerate(fig.data):
                trace.visible = vis[index]

            fig.layout.title = (
                f"{sector_dd.value} - Count of {selected_cap} Stocks Above Sharpe Ratio Z-Score Thresholds "
                f"(200-Day Window)<br><sup>{thresh_dd.value}</sup>"
            )

            fig.layout.shapes = shapes
            fig.layout.annotations = annotations
            fig.layout.xaxis.type = "date"

    sector_dd.observe(apply_selection, names="value")
    thresh_dd.observe(apply_selection, names="value")

    fig.update_layout(
        title=f"{sectors[0]} - Count of {selected_cap} Stocks Above Sharpe Ratio Z-Score Thresholds",
        xaxis_title="Date",
        yaxis_title="Number of Stocks",
        xaxis=dict(type="date"),
        hovermode="x unified",
        height=height,
    )

    apply_selection()

    return VBox([HBox([sector_dd, thresh_dd]), fig])

#function to plot the results for count of stocks above/below Sharpe above benchmark


In [ ]:
# Block 9

import socket
import numpy as np
import plotly.graph_objects as go
from dash import Dash, Input, Output, State, ctx, dcc, html, no_update

FONT_FAMILY = '\"IBM Plex Sans\", \"Segoe UI\", sans-serif'
ACCENT_COLOR = "#0f766e"
SECONDARY_COLOR = "#1d4ed8"
GRID_COLOR = "rgba(148, 163, 184, 0.22)"
PAPER_COLOR = "#ffffff"
PLOT_COLOR = "#ffffff"
TEXT_COLOR = "#0f172a"
MUTED_TEXT_COLOR = "#475569"
BORDER_COLOR = "#d9dee5"


def get_available_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def build_market_profile_summary(sector_label, threshold_label, selected_cap):
    summary_items = [
        ("Sector", sector_label),
        ("Capitalization", selected_cap),
        ("Threshold", threshold_label.replace(" (window: 200)", "")),
    ]

    return [
        html.Div(
            [
                html.Span(f"{label}:", style={"color": MUTED_TEXT_COLOR}),
                html.Span(value, style={"fontWeight": "600", "color": TEXT_COLOR}),
            ],
            style={"display": "flex", "gap": "6px", "alignItems": "baseline"},
        )
        for label, value in summary_items
    ]


def apply_market_profile_figure_style(fig, title, x_range=None):
    fig.update_xaxes(title_text="Date", type="date", showgrid=True, gridcolor=GRID_COLOR)
    if x_range:
        fig.update_xaxes(range=x_range)
    else:
        fig.update_xaxes(autorange=True)

    fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR)
    fig.update_layout(
        title=dict(text=title, x=0, xanchor="left", font=dict(size=18)),
        hovermode="x unified",
        template="plotly_white",
        paper_bgcolor=PAPER_COLOR,
        plot_bgcolor=PLOT_COLOR,
        margin=dict(t=64, r=24, b=48, l=60),
        height=340,
        font=dict(color=TEXT_COLOR, family=FONT_FAMILY),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        uirevision="market-profile-sync",
    )
    return fig


def build_benchmark_figure(benchmark_counts_df, sector_label, x_range=None):
    net_count_series = benchmark_counts_df["Net Count Above Benchmark Sharpe"]
    x_values = normalize_x_index(net_count_series.index)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=net_count_series,
            mode="lines",
            name="Net Count",
            line=dict(color=ACCENT_COLOR, width=2.5),
            hovertemplate=(
                f"{sector_label}"
                f"<br>Date: %{{x}}"
                f"<br>Net Count: %{{y}}"
                f"<extra></extra>"
            ),
        )
    )

    net_count_mean = float(net_count_series.mean())
    net_count_std = float(net_count_series.std())
    net_count_levels = [
        (net_count_mean, "Mean", "rgba(15, 23, 42, 0.45)"),
        (net_count_mean + net_count_std, "+1 sigma", "rgba(15, 118, 110, 0.45)"),
        (net_count_mean - net_count_std, "-1 sigma", "rgba(15, 118, 110, 0.45)"),
        (net_count_mean + 2 * net_count_std, "+2 sigma", "rgba(37, 99, 235, 0.42)"),
        (net_count_mean - 2 * net_count_std, "-2 sigma", "rgba(37, 99, 235, 0.42)"),
        (net_count_mean + 3 * net_count_std, "+3 sigma", "rgba(220, 38, 38, 0.35)"),
        (net_count_mean - 3 * net_count_std, "-3 sigma", "rgba(220, 38, 38, 0.35)"),
    ]
    for level, label, color in net_count_levels:
        fig.add_hline(
            y=level,
            line_dash="dash",
            line_width=1,
            line_color=color,
            annotation_text=f"{label}: {level:.2f}",
            annotation_position="top right",
        )

    fig.update_yaxes(title_text="Net Count")
    return apply_market_profile_figure_style(
        fig,
        f"Net Count Above Benchmark Sharpe | {sector_label}",
        x_range=x_range,
    )


def build_zscore_figure(
    zscore_counts_df,
    threshold_label,
    sector_label,
    selected_cap,
    x_range=None,
    sigmas=(0.5, 1, 1.5, 2),
    show_mean=True,
    ddof=0,
):
    zscore_series = zscore_counts_df[threshold_label]
    x_values = normalize_x_index(zscore_series.index)
    zscore_values = np.asarray(zscore_series.values, dtype=float)
    zscore_clean = zscore_values[~np.isnan(zscore_values)]

    if zscore_clean.size == 0:
        mu, sd = 0.0, 0.0
    else:
        mu = float(np.mean(zscore_clean))
        sd = float(np.std(zscore_clean, ddof=ddof))

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=zscore_series,
            mode="lines",
            name=threshold_label,
            line=dict(color=SECONDARY_COLOR, width=2.5),
            hovertemplate=(
                f"{sector_label}"
                f"<br>Threshold: {threshold_label}"
                f"<br>Date: %{{x}}"
                f"<br>Count: %{{y}}"
                f"<extra></extra>"
            ),
        )
    )

    if show_mean:
        fig.add_hline(
            y=mu,
            line_width=2,
            line_color="rgba(15, 23, 42, 0.55)",
            annotation_text=f"mu: {mu:.2f}",
            annotation_position="top right",
        )

    if sd > 0:
        fig.add_hrect(
            y0=mu + 1.5 * sd,
            y1=mu + 2 * sd,
            fillcolor="rgba(248, 113, 113, 0.16)",
            line_width=0,
        )

        for sigma in sigmas:
            y_level = mu + sigma * sd
            fig.add_hline(
                y=y_level,
                line_dash="dash",
                line_width=1,
                line_color="rgba(37, 99, 235, 0.4)",
                annotation_text=f"+{sigma} sigma ({y_level:.2f})",
                annotation_position="top right",
            )

    fig.update_yaxes(title_text="Number of Stocks")
    return apply_market_profile_figure_style(
        fig,
        (
            f"{sector_label} | Count of {selected_cap} Stocks Above Sharpe Ratio Z-Score Thresholds"
            f"<br><sup>{threshold_label}</sup>"
        ),
        x_range=x_range,
    )


def build_dashboard_card(title, graph_id):
    return html.Div(
        [
            html.Div(
                title,
                style={"fontSize": "16px", "fontWeight": "600", "color": TEXT_COLOR, "marginBottom": "8px"},
            ),
            dcc.Graph(
                id=graph_id,
                config={"displayModeBar": False, "responsive": True},
                style={"height": "360px"},
            ),
        ],
        style={"paddingTop": "12px", "borderTop": f"1px solid {BORDER_COLOR}"},
    )


def extract_synced_x_range(relayout_data):
    if not relayout_data:
        return no_update

    if relayout_data.get("xaxis.autorange"):
        return {"x_range": None}

    if "xaxis.range[0]" in relayout_data and "xaxis.range[1]" in relayout_data:
        return {"x_range": [relayout_data["xaxis.range[0]"], relayout_data["xaxis.range[1]"]]}

    if "xaxis.range" in relayout_data:
        return {"x_range": relayout_data["xaxis.range"]}

    return no_update


sector_dashboard_config = {
    "all_companies": {
        "label": "All Companies",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_all,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200,
    },
    "health_care": {
        "label": "Health Care",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_healthcare_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_healthcare_large_cap,
    },
    "information_technology": {
        "label": "Information Technology",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_technology_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_technology_large_cap,
    },
    "financials": {
        "label": "Financials",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_financials_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_financials_large_cap,
    },
    "consumer_discretionary": {
        "label": "Consumer Discretionary",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_consumer_discretionary_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_consumer_discretionary_large_cap,
    },
    "consumer_staples": {
        "label": "Consumer Staples",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_consumer_staples_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_consumer_staples_large_cap,
    },
    "energy": {
        "label": "Energy",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_energy_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_energy_large_cap,
    },
    "industrials": {
        "label": "Industrials",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_industrials_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_industrials_large_cap,
    },
    "materials": {
        "label": "Materials",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_materials_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_materials_large_cap,
    },
    "real_estate": {
        "label": "Real Estate",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_real_estate_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_real_estate_large_cap,
    },
    "utilities": {
        "label": "Utilities",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_utilities_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_utilities_large_cap,
    },
    "communication_services": {
        "label": "Communication Services",
        "benchmark_counts": count_stocks_above_below_sharpe_benchmark_communication_services_large_cap,
        "zscore_counts": count_stocks_above_below_sharpe_zscore_200_communication_services_large_cap,
    },
}

thresholds_list = [
    "Count Above Z-Score 0 (window: 200)",
    "Count Above Z-Score 1 (window: 200)",
    "Count Above Z-Score 2 (window: 200)",
    "Count Above Z-Score 3 (window: 200)",
    "Count Below Z-Score -0 (window: 200)",
    "Count Below Z-Score -1 (window: 200)",
    "Count Below Z-Score -2 (window: 200)",
    "Count Below Z-Score -3 (window: 200)",
]

market_profile_app = Dash("market_profile_block_9")
market_profile_app.layout = html.Div(
    [
        dcc.Store(id="market-profile-x-range"),
        html.H2("Market Profile Dashboard", style={"margin": "0 0 6px 0", "color": TEXT_COLOR}),
        html.Div(
            "Zoom or pan either chart and the other will follow the same date range.",
            style={"marginBottom": "12px", "color": MUTED_TEXT_COLOR},
        ),
        html.Div(
            id="market-profile-summary",
            style={"display": "flex", "gap": "18px", "flexWrap": "wrap", "marginBottom": "14px"},
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Sector", style={"fontWeight": "600", "color": TEXT_COLOR}),
                        dcc.Dropdown(
                            id="sector-dropdown",
                            options=[
                                {"label": config["label"], "value": sector_key}
                                for sector_key, config in sector_dashboard_config.items()
                            ],
                            value="all_companies",
                            clearable=False,
                        ),
                    ],
                    style={"minWidth": "280px", "flex": "1"},
                ),
                html.Div(
                    [
                        html.Label("Threshold", style={"fontWeight": "600", "color": TEXT_COLOR}),
                        dcc.Dropdown(
                            id="threshold-dropdown",
                            options=[{"label": threshold, "value": threshold} for threshold in thresholds_list],
                            value=thresholds_list[0],
                            clearable=False,
                        ),
                    ],
                    style={"minWidth": "280px", "flex": "1"},
                ),
            ],
            style={
                "display": "flex",
                "gap": "12px",
                "flexWrap": "wrap",
                "paddingBottom": "12px",
                "borderBottom": f"1px solid {BORDER_COLOR}",
                "marginBottom": "12px",
            },
        ),
        build_dashboard_card("Benchmark Breadth", "market-profile-benchmark-graph"),
        build_dashboard_card("Z-Score Participation", "market-profile-zscore-graph"),
    ],
    style={"padding": "16px 0", "background": PAPER_COLOR, "color": TEXT_COLOR, "fontFamily": FONT_FAMILY},
)


@market_profile_app.callback(
    Output("market-profile-summary", "children"),
    Output("market-profile-benchmark-graph", "figure"),
    Output("market-profile-zscore-graph", "figure"),
    Input("sector-dropdown", "value"),
    Input("threshold-dropdown", "value"),
    Input("market-profile-x-range", "data"),
)
def update_market_profile_dashboard(sector_key, threshold_label, x_range_data):
    config = sector_dashboard_config[sector_key]
    sector_label = config["label"]
    x_range = x_range_data.get("x_range") if x_range_data else None

    return (
        build_market_profile_summary(
            sector_label=sector_label,
            threshold_label=threshold_label,
            selected_cap=selected_capitalization,
        ),
        build_benchmark_figure(
            benchmark_counts_df=config["benchmark_counts"],
            sector_label=sector_label,
            x_range=x_range,
        ),
        build_zscore_figure(
            zscore_counts_df=config["zscore_counts"],
            threshold_label=threshold_label,
            sector_label=sector_label,
            selected_cap=selected_capitalization,
            x_range=x_range,
        ),
    )


@market_profile_app.callback(
    Output("market-profile-x-range", "data"),
    Input("market-profile-benchmark-graph", "relayoutData"),
    Input("market-profile-zscore-graph", "relayoutData"),
    State("market-profile-x-range", "data"),
    prevent_initial_call=True,
)
def sync_market_profile_graph_ranges(benchmark_relayout, zscore_relayout, current_range_data):
    relayout_data = benchmark_relayout if ctx.triggered_id == "market-profile-benchmark-graph" else zscore_relayout
    synced_range = extract_synced_x_range(relayout_data)

    if synced_range is no_update or synced_range == current_range_data:
        return no_update

    return synced_range


market_profile_app.run(
    jupyter_mode="inline",
    port=get_available_port(),
    debug=False,
)


In [ ]:
# Block 10

#load broad market data
#each of these is a dictionary that holds multiple dataframes with the relevant data with 'Close', 'Open', 'High', 'Low', 'Volume' columns

major_quity_indices_data = md.get_major_equity_indices_data() #MAJOR INDICES
dividends_data = md.get_dividend_data() #DIVIDENDS
sector         = md.get_sector_data() #SECTOR
factors_data   = md.get_factor_data() #FACTORS
beta           = md.get_beta_factors() #BETA
size_vs_value  = md.get_size_vs_value_data() #SIZE VS. VALUE


#give me the close for dividends_data and convert to a dataframe
major_equity_indices_close = pd.DataFrame({symbol: data['Close'] for symbol, data in major_quity_indices_data.items()})
dividends_close = pd.DataFrame({symbol: data['Close'] for symbol, data in dividends_data.items()})
sector_close = pd.DataFrame({symbol: data['Close'] for symbol, data in sector.items()})
factors_close = pd.DataFrame({symbol: data['Close'] for symbol, data in factors_data.items()})
beta_close = pd.DataFrame({symbol: data['Close'] for symbol, data in beta.items()})
size_vs_value_close = pd.DataFrame({symbol: data['Close'] for symbol, data in size_vs_value.items()})

#now that i have cloe prices, convert to n day sharpe

def calculate_sharpe_ratios(df, window=200, risk_free_rate=0.0):
    """
    Calculate rolling Sharpe ratios for each column in the DataFrame.

    Parameters:
    - df: DataFrame with datetime index and columns representing different assets.
    - window: Lookback period for rolling calculations (default: 200).
    - risk_free_rate: Annualized risk-free rate (default: 0.0).

    Returns:
    - DataFrame of rolling Sharpe ratios with the same shape as input df.
    """
    # Calculate daily returns
    daily_returns = df.pct_change().dropna()

    # Annualize the risk-free rate for daily calculations
    daily_risk_free_rate = risk_free_rate / 252

    # Calculate excess returns by subtracting the daily risk-free rate
    excess_returns = daily_returns - daily_risk_free_rate

    # Calculate rolling mean and std of excess returns
    rolling_mean = excess_returns.rolling(window=window).mean()
    rolling_std = excess_returns.rolling(window=window).std()

    # Calculate rolling Sharpe ratio
    sharpe_ratios = rolling_mean / rolling_std

    return sharpe_ratios

#calculate sharpe ratios for each of the datasets
major_equity_indices_sharpe = calculate_sharpe_ratios(major_equity_indices_close)
dividends_sharpe = calculate_sharpe_ratios(dividends_close)
sector_sharpe = calculate_sharpe_ratios(sector_close)
factors_sharpe = calculate_sharpe_ratios(factors_close)
beta_sharpe = calculate_sharpe_ratios(beta_close)
size_vs_value_sharpe = calculate_sharpe_ratios(size_vs_value_close)

#now that i have the sharpe ratios, calculate the z-scores for each of the datasets
def calculate_sharpe_zscores(sharpe_df, window=200):
    """
    Calculate rolling z-scores of Sharpe ratios for each column in the DataFrame.

    Parameters:
    - sharpe_df: DataFrame of rolling Sharpe ratios with datetime index and asset columns.
    - window: Lookback period for rolling mean and std calculations (default: 200).

    Returns:
    - DataFrame of rolling z-scores with the same shape as input sharpe_df.
    """
    # Calculate rolling mean and std of Sharpe ratios
    rolling_mean = sharpe_df.rolling(window=window).mean()
    rolling_std = sharpe_df.rolling(window=window).std()

    # Calculate z-scores
    z_scores = (sharpe_df - rolling_mean) / rolling_std

    return z_scores

major_equity_indices_sharpe_zscore = calculate_sharpe_zscores(major_equity_indices_sharpe)
dividends_sharpe_zscore = calculate_sharpe_zscores(dividends_sharpe)
sector_sharpe_zscore = calculate_sharpe_zscores(sector_sharpe)
factors_sharpe_zscore = calculate_sharpe_zscores(factors_sharpe)
beta_sharpe_zscore = calculate_sharpe_zscores(beta_sharpe)
size_vs_value_sharpe_zscore = calculate_sharpe_zscores(size_vs_value_sharpe)





In [ ]:
# Block 11

#plot 
def plot_df(data, title):
    fig = go.Figure()
    for column in data.columns:
        fig.add_trace(go.Scatter(x=data.index, y=data[column], mode='lines', name=column))
    fig.update_layout(title=title, xaxis_title='Date', yaxis_title='Close Price')
    return fig


#plot_df but choose between columns to plot with a dropdown menu, add +/- 1,2,3 sigma lines based on the mean and std of the selected column, and add annotations for the sigma lines on the right side of the plot
def plot_df_with_dropdown(data, title):
    fig = go.Figure()
    columns = data.columns.tolist()
    for column in columns:
        fig.add_trace(go.Scatter(x=data.index, y=data[column], mode='lines', name=column, visible=(column == columns[0])))
    # Add dropdown menu
    fig.update_layout(
        title=title,
        xaxis_title='Date',
        yaxis_title='Close Price',
        updatemenus=[
            dict(
                buttons=[
                    dict(
                        label=col,
                        method='update',
                        args=[{'visible': [c == col for c in columns]},
                              {'title': f"{title} - {col}"}]
                    ) for col in columns
                ],
                direction='down',
                showactive=True,
                x=0.01,
                y=1.15,
                xanchor='left',
                yanchor='top'
            )
        ]
    )
    #add sigma lines and annotations for the first column by default
    first_col = columns[0]
    mean = data[first_col].mean()
    std = data[first_col].std()
    for k in range(1, 4):
        fig.add_shape(type='line', x0=data.index.min(), x1=data.index.max(), y0=mean + k*std, y1=mean + k*std, line=dict(dash='dash', width=1), visible=True)
        fig.add_shape(type='line', x0=data.index.min(), x1=data.index.max(), y0=mean - k*std, y1=mean - k*std, line=dict(dash='dash', width=1), visible=True)
        fig.add_annotation(x=data.index.max(), y=mean + k*std, text=f"+{k}Ïƒ ({mean + k*std:.2f})", showarrow=False, xanchor='left', yanchor='middle', visible=True)
        fig.add_annotation(x=data.index.max(), y=mean - k*std, text=f"-{k}Ïƒ ({mean - k*std:.2f})", showarrow=False, xanchor='left', yanchor='middle', visible=True)
        
        
    return fig
plot_df_with_dropdown(major_equity_indices_sharpe_zscore, "Major Equity Indices Close Prices").show()
plot_df_with_dropdown(dividends_sharpe_zscore, "Dividends Close Prices").show()
plot_df_with_dropdown(sector_sharpe_zscore, "Sector Close Prices").show()
plot_df_with_dropdown(factors_sharpe_zscore, "Factors Close Prices").show()
plot_df_with_dropdown(beta_sharpe_zscore, "Beta Close Prices").show()
plot_df_with_dropdown(size_vs_value_sharpe_zscore, "Size vs Value Close Prices").show()


In [ ]:
# Block 12

indices_data = md.get_indices_data()
sector_data = md.get_sector_data()
factor_data = {key: md.get_factor_data()[key] for key in ['Quality', 'Value', 'Growth', 'Momentum', 'Market Capitalization', 'Low Beta', 'High Beta', 'Minimum_Volatility_Value']}

market_cap_data = md.get_market_cap_data()

#time_frame_short = 21
#time_frame_mid   = 50
time_frame_long = 252

benchmark = 'S&P 500'  # S&P 500 ETF as benchmark


In [ ]:
# Block 13

#Market Cap Analysis

def plot_combined_returns_differences(time_frame):
    fig = make_subplots(rows=3, cols=1, subplot_titles=[
        f'Z-Score of Sharpe Ratio for Large Cap over {time_frame} Days',
        f'Z-Score of Sharpe Ratio Difference between Mid Cap and Small Cap over {time_frame} Days',
        f'Z-Score of Sharpe Ratio Difference between Large, Mid, and Small Cap over {time_frame} Days'
    ])
    
    # Compute daily returns
    large_cap_daily_returns = market_cap_data['Large Cap']['Close'].pct_change()
    mid_cap_daily_returns = market_cap_data['Mid Cap']['Close'].pct_change()
    small_cap_daily_returns = market_cap_data['Small Cap']['Close'].pct_change()
    
    # Compute rolling Sharpe ratios
    def rolling_sharpe(returns, window, rf):
        rolling_mean = returns.rolling(window=window).mean()
        rolling_std = returns.rolling(window=window).std()
        return (rolling_mean - rf) / rolling_std
    
    large_cap_sharpe = rolling_sharpe(large_cap_daily_returns, time_frame, risk_free_rate)
    mid_cap_sharpe = rolling_sharpe(mid_cap_daily_returns, time_frame, risk_free_rate)
    small_cap_sharpe = rolling_sharpe(small_cap_daily_returns, time_frame, risk_free_rate)
    
    # First subplot: Z-Score of Large Cap Sharpe Ratio
    mean_large_sharpe = large_cap_sharpe.mean()
    std_large_sharpe = large_cap_sharpe.std()
    z_score_large_sharpe = (large_cap_sharpe - mean_large_sharpe) / std_large_sharpe
    
    fig.add_trace(go.Scatter(x=z_score_large_sharpe.index, y=z_score_large_sharpe, mode='lines', name='Large Cap Sharpe Ratio (Z-Score)'), row=1, col=1)
    
    # Second subplot: Mid Cap - Small Cap Sharpe difference
    sharpe_difference_mid_small = mid_cap_sharpe - small_cap_sharpe
    mean_diff = sharpe_difference_mid_small.mean()
    std_diff = sharpe_difference_mid_small.std()
    z_score = (sharpe_difference_mid_small - mean_diff) / std_diff
    
    fig.add_trace(go.Scatter(x=z_score.index, y=z_score, mode='lines', name='Mid Cap - Small Cap Sharpe Difference (Z-Score)'), row=2, col=1)
    
    # Third subplot: Large - Mid and Large - Small Sharpe differences
    sharpe_difference_large_mid = large_cap_sharpe - mid_cap_sharpe
    mean_diff_large_mid = sharpe_difference_large_mid.mean()
    std_diff_large_mid = sharpe_difference_large_mid.std()
    z_score_large_mid = (sharpe_difference_large_mid - mean_diff_large_mid) / std_diff_large_mid
    
    fig.add_trace(go.Scatter(x=z_score_large_mid.index, y=z_score_large_mid, mode='lines', name='Large Cap - Mid Cap Sharpe Difference (Z-Score)'), row=3, col=1)
    
    sharpe_difference_large_small = large_cap_sharpe - small_cap_sharpe
    mean_diff_large_small = sharpe_difference_large_small.mean()
    std_diff_large_small = sharpe_difference_large_small.std()
    z_score_large_small = (sharpe_difference_large_small - mean_diff_large_small) / std_diff_large_small
    
    fig.add_trace(go.Scatter(x=z_score_large_small.index, y=z_score_large_small, mode='lines', name='Large Cap - Small Cap Sharpe Difference (Z-Score)'), row=3, col=1)
    
    fig.update_layout(template='plotly_dark')
    #change height of the figure
    fig.update_layout(height=1300)
    
    # Add horizontal lines and shapes for first subplot (mirroring second)
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)
    fig.add_hline(y=1, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=-1, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=2, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=-2, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=3, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=-3, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_shape(type="rect", x0=z_score_large_sharpe.index.min(), x1=z_score_large_sharpe.index.max(), y0=-2, y1=-1, fillcolor="green", opacity=0.2, line_width=0, row=1, col=1)
    fig.add_shape(type="rect", x0=z_score_large_sharpe.index.min(), x1=z_score_large_sharpe.index.max(), y0=1, y1=2, fillcolor="red", opacity=0.2, line_width=0, row=1, col=1)
    fig.add_annotation(x=z_score_large_sharpe.index[len(z_score_large_sharpe)//2], y=1.5, text="Bearish Large Cap", showarrow=False, font=dict(color="red", size=12), row=1, col=1)
    fig.add_annotation(x=z_score_large_sharpe.index[len(z_score_large_sharpe)//2], y=-1.5, text="Long Large Cap", showarrow=False, font=dict(color="green", size=12), row=1, col=1)
    
    # Add horizontal lines and shapes for middle subplot
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=1)
    fig.add_hline(y=1, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_hline(y=-1, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_hline(y=2, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_hline(y=-2, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_hline(y=3, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_hline(y=-3, line_dash="dot", line_color="gray", row=2, col=1)
    fig.add_shape(type="rect", x0=z_score.index.min(), x1=z_score.index.max(), y0=-2, y1=-1, fillcolor="gray", opacity=0.2, line_width=0, row=2, col=1)
    fig.add_shape(type="rect", x0=z_score.index.min(), x1=z_score.index.max(), y0=1, y1=2, fillcolor="gray", opacity=0.2, line_width=0, row=2, col=1)
    fig.add_annotation(x=z_score.index[len(z_score)//2], y=1.5, text="Long Mid Cap / Short Small Cap", showarrow=False, font=dict(color="gray", size=12), row=2, col=1)
    fig.add_annotation(x=z_score.index[len(z_score)//2], y=-1.5, text="Long Small Cap / Short Mid Cap", showarrow=False, font=dict(color="gray", size=12), row=2, col=1)
    
    # Add horizontal line for bottom subplot
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=3, col=1)
    
    fig.show()

plot_combined_returns_differences(time_frame_long)


In [ ]:
# Block 14

import math

#plot sector data 
#sector data is a dictionary with sector names as keys and dataframes as values

def plot_sector_performance(time_frame):
    sectors = list(sector_data.keys())
    num_sectors = len(sectors)
    
    # Calculate grid dimensions (e.g., roughly square grid)
    cols = math.ceil(math.sqrt(num_sectors))
    rows = math.ceil(num_sectors / cols)
    
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[
        f'{sector} Sector Z-Score over {time_frame} Days' for sector in sectors
    ])
    
    for i, (sector, data) in enumerate(sector_data.items()):
        row = (i // cols) + 1
        col = (i % cols) + 1
        
        daily_returns = data['Close'].pct_change()
        
        # Compute rolling Sharpe ratio
        def rolling_sharpe(returns, window, rf):
            rolling_mean = returns.rolling(window=window).mean()
            rolling_std = returns.rolling(window=window).std()
            return (rolling_mean - rf) / rolling_std
        
        sharpe_ratio = rolling_sharpe(daily_returns, time_frame, risk_free_rate)
        
        mean_sharpe = sharpe_ratio.mean()
        std_sharpe = sharpe_ratio.std()
        z_score = (sharpe_ratio - mean_sharpe) / std_sharpe
        
        fig.add_trace(go.Scatter(x=z_score.index, y=z_score, mode='lines', name=f'{sector} Sector (Z-Score)'), row=row, col=col)
        fig.add_hline(y=0, line_dash="dash", line_color="red", row=row, col=col)
        fig.add_hline(y=1, line_dash="dot", line_color="gray", row=row, col=col)
        fig.add_hline(y=-1, line_dash="dot", line_color="gray", row=row, col=col)
        fig.add_hline(y=2, line_dash="dot", line_color="gray", row=row, col=col)
        fig.add_hline(y=-2, line_dash="dot", line_color="gray", row=row, col=col)
        fig.add_hline(y=3, line_dash="dot", line_color="gray", row=row, col=col)
        fig.add_hline(y=-3, line_dash="dot", line_color="gray", row=row, col=col)
    
    fig.update_layout(template='plotly_dark', height=400 * rows)
    fig.show()



plot_sector_performance(time_frame_long)


In [ ]:
# Block 15

#plot factor data on one chart

def plot_factor_performance(time_frame):
    fig = make_subplots(rows=1, cols=1, subplot_titles=[
        f'Benchmark Minus Factor Z-Scores over {time_frame} Days'
    ])
    
    # Get benchmark data (assuming 'SPY' as benchmark)
    benchmark_data = indices_data['S&P 500']
    benchmark_daily_returns = benchmark_data['Close'].pct_change()
    
    # Compute rolling Sharpe ratio for benchmark
    def rolling_sharpe(returns, window, rf):
        rolling_mean = returns.rolling(window=window).mean()
        rolling_std = returns.rolling(window=window).std()
        return (rolling_mean - rf) / rolling_std
    
    benchmark_sharpe = rolling_sharpe(benchmark_daily_returns, time_frame, risk_free_rate)
    benchmark_mean_sharpe = benchmark_sharpe.mean()
    benchmark_std_sharpe = benchmark_sharpe.std()
    benchmark_z_score = (benchmark_sharpe - benchmark_mean_sharpe) / benchmark_std_sharpe
    
    for factor, data in factor_data.items():
        daily_returns = data['Close'].pct_change()
        
        sharpe_ratio = rolling_sharpe(daily_returns, time_frame, risk_free_rate)
        
        mean_sharpe = sharpe_ratio.mean()
        std_sharpe = sharpe_ratio.std()
        z_score = (sharpe_ratio - mean_sharpe) / std_sharpe
        
        # Compute benchmark minus factor z-score
        diff_z_score = benchmark_z_score - z_score
        
        fig.add_trace(go.Scatter(x=diff_z_score.index, y=diff_z_score, mode='lines', name=f'Benchmark - {factor}'), row=1, col=1)
    
    fig.update_layout(template='plotly_dark', height=600)
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)
    fig.show()
plot_factor_performance(time_frame_long)


In [ ]:
# Block 16

import pandas_datareader.data as web
import yfinance as yf
import datetime
import pandas as pd
import statsmodels.api as sm

# Function to simplify datetime index format


# Define the stock ticker
ticker = "SPY"  # You can change to your stock of choice

# Define period and interval
period = 'max'  # Maximum period to get historical data
interval = '1d'

# Fetch stock's historical closing prices
ticker_close = yf.Ticker(ticker).history(period=period, interval=interval)['Close']
ticker_close = simplify_datetime_index(ticker_close)

# Define start and end dates for Fama-French data
start = datetime.datetime(1963, 1, 1)  # Daily data is available from 1963 onward
end = datetime.datetime.today()

# Fetch **daily** Fama-French 5-factor data
ff5_daily = web.DataReader('F-F_Research_Data_5_Factors_2x3_Daily', 'famafrench', start, end)

# Convert Fama-French data to pandas datetime
ff5_daily = ff5_daily[0].reset_index()
ff5_daily['Date'] = pd.to_datetime(ff5_daily['Date'], format='%Y%m%d')
ff5_daily = ff5_daily.set_index('Date')

# Align the data (Ensure both stock and Fama-French data have the same date range)
common_index = ff5_daily.index.intersection(ticker_close.index)
ff5_daily = ff5_daily.loc[common_index]
ticker_close = ticker_close.loc[common_index]

# Convert daily data to monthly data (resample to monthly frequency)
ff5_monthly = ff5_daily.resample('M').last()
ticker_close_monthly = ticker_close.resample('M').last()

# Calculate monthly returns for the stock (percentage change)
ticker_returns = ticker_close_monthly.pct_change().dropna()

# Calculate excess returns (stock return - risk free rate)
rf = ff5_monthly['RF'] / 100  # Convert risk-free rate from percentage to decimal
excess_stock_returns = ticker_returns - rf

# Ensure data is aligned by date (final check for common dates between stock returns and Fama-French data)
common_dates = ticker_returns.index.intersection(ff5_monthly.index)
print(f"Found {len(common_dates)} common dates between stock returns and Fama-French data")

# Align the datasets (both Fama-French and excess returns)
ff5_monthly = ff5_monthly.loc[common_dates]
excess_stock_returns = excess_stock_returns.loc[common_dates]

# Prepare for the Fama-French 5-factor regression
X = ff5_monthly[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].copy() / 100  # Convert from percentage to decimal
X = sm.add_constant(X)  # Add constant for alpha (intercept term)
y = excess_stock_returns

# Print shapes of X and y to check if they align properly
print(f"Shape of X (Fama-French factors): {X.shape}")
print(f"Shape of y (Excess stock returns): {y.shape}")

# Perform the OLS regression
model = sm.OLS(y, X)

# Fit the model
results = model.fit()

# Get the regression results
print(results.summary())
